# Introduction 🎇
We all know the importance of wearing seatbelts while driving, as they can prevent serious injuries in the event of an accident. However, many drivers still neglect to wear them, either due to forgetfulness, complacency, or even ignorance of the law. This is where our seatbelt classifier comes in - it uses deep learning algorithms to analyze video footage from a vehicle's dashboard camera, and quickly determine whether the driver or passenger is wearing their seatbelt.

The system is easy to use, and can be installed in any vehicle with a dashboard camera. Once installed, it continuously monitors the video feed and alerts the driver if it detects that they are not wearing their seatbelt. This can help to prevent accidents, reduce injuries, and save lives.

According to the MoRTH(Ministry of Road Transport and Highways of India) report, a **total of 16,397 people lost their lives for not wearing seat belts**. The data shows why wearing seat belts on the backseat of a car is equally important.[refrence](https://auto.hindustantimes.com/auto/news/not-wearing-seat-belts-caused-death-of-16-397-people-in-road-accidents-last-year-41672376310364.html)

# Problem 📒
Although car manufacturers have given some sort of a buzzer that beeps when the driver is not wearing a seatbelt but that can be disabled by using a **fake seatbelt buckle** selling openly in India (yup paying ₹300 or $3.62 to kill yourself but can’t wear a company fitted seatbelt )
![buckel](https://cdn.discordapp.com/attachments/713708382238015510/1079452174062460998/image.png)
# Request 🙏
if you find such thing in someone's car plz throw it away.



# Importing Libraries 🎃

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
import os
from keras.models import load_model 
import tensorflow as tf
import os
%matplotlib inline

# Data 📄
I have collected data from youtuber who make content related to cars.

**`seatbelt`**-contain yolov5 model + labeled data .

**`seatbelt2`**-contain data for training of a seatbelt vs no seatbelt classifier.

**`seatbelt3`**-contain both trained yolov5,classifier and test video.

Though I could not get enough data for training 😢, that’s probably why Yolo's classifier was not able to perform well.

I used data argumentation to increase the sample size and that worker.`

## Setting up W&B 

In [ ]:
!wandb login #insert your wandb key here

In [ ]:
!7z  x  -pyolo  //kaggle/input/seatbelt #unzipping the zip file 

# Training of yolo model 🤖
Followed this [reop](https://github.com/ultralytics/yolov5) for training of my yolo model

In [ ]:
cd /kaggle/working/yolov5

In [ ]:
!pip install -r requirements.txt   #most of the library are installed except one thop

In [ ]:
#defining path for training an val data 
text="""
path: ../data
train: train/images
val: val/images
# number of classes
nc: 1

# class names
names: ['seatbelt']
"""
f=open('data.yaml','w')
f.write(text)
#no idea about below output 

In [ ]:
#rececking wheather data.yaml contain right location of path
f=open('data.yaml')
f.readlines()


In [ ]:
!python train.py --img 320 --batch 8 --epochs 100 --data data.yaml --weights yolov5n.pt  --device 0 

# Classifier trining 🤖
I trained classifier at [teachable Machine](https://teachablemachine.withgoogle.com/train) using **seatbelt2 data** and it turned out to be a great model .

# Testing time 📄

#### ⚠ Refer to [this](https://docs.ultralytics.com/reference/results/) docs for more info of about model.result()

In [ ]:
def prediction_func(img):
    img=cv2.resize(img,(224, 224),interpolation=cv2.INTER_AREA)
    img=(img/127.5)-1
    img=tf.expand_dims(img,axis=0)
    pred=predictor.predict(img)
    index = np.argmax(pred)
    class_name = class_names[index]
    confidence_score = pred[0][index]
    return class_name,confidence_score

In [ ]:
class_names={0:'noseatblet',1:'seatblet'}
path="/kaggle/working/data/val/images/"
predictor = load_model("/kaggle/input/seatbelt3/seatbelt/converted_keras/keras_model.h5", compile=False)

In [ ]:
model = torch.hub.load("ultralytics/yolov5", "custom",path='/kaggle/input/seatbelt3/seatbelt/best.pt',force_reload=True)  # or yolov5n - yolov5x6, custom

In [ ]:
images=os.listdir(path)
images

In [ ]:
# Testing with classifier
for i in images:
    img=cv2.imread(path+i)
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    results = model(img)
    boxes=results.xyxy[0]
    boxes=boxes.cpu()
    for j in boxes:
        x1,y1,x2,y2,prob,clss=j.numpy()
        x1,y1,x2,y2=int(x1),int(y1),int(x2),int(y2)
        img_crop=img[y1:y2,x1:x2]
        clss,prob=prediction_func(img_crop)
        cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(img,f'{clss} {str(prob)[:4]}',(x1-10,y1-10),cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)
    plt.imshow(img)
    plt.show()
    print("")

In [ ]:
cap=cv2.VideoCapture('/kaggle/input/seatbelt3/seatbelt/test_Trim.mp4')
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
   
size = (frame_width, frame_height)

writer = cv2.VideoWriter('test.mp4', 
                         cv2.VideoWriter_fourcc(*'MP4V'),
                         30,size)
frame=900
count=0
while True:
    ret,img=cap.read()
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    results = model(img)
    boxes=results.xyxy[0]
    boxes=boxes.cpu()
    for j in boxes:
        x1,y1,x2,y2,prob,clss=j.numpy()
        x1,y1,x2,y2=int(x1),int(y1),int(x2),int(y2)
        img_crop=img[y1:y2,x1:x2]
        clss,prob=prediction_func(img_crop)
        cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(img,f'{clss} {str(prob)[:4]}',(x1-10,y1-10),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)
    img=cv2.cvtColor(img,cv2.COLOR_RGB2BGR)
    writer.write(img)
    if count>frame:
        break
    count+=1
cap.release()
writer.release()

In [ ]:
from IPython.display import HTML

HTML('<div align="center"><iframe align = "middle" width="790" height="440" src="https://www.youtube.com/embed/KSyJo6f3bgk" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe></div>')


# Conclusion 🖐
1. Yolo object detection is working very good considering the amount of data provided for its training.
2. The classifier on the other hand is not so good in itentifying the correct class that can be due to less data provided to it for training.

# My learining 💡
* Train a yolo Model
* Data collection and labelling
* Data augmentation

